In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# chain = prompt | llm | parser
# LCEL : LangChain Expression Language

In [ ]:
# RunnableSequence
# RunnableParallel
# RunnablePassthrough

In [2]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    summary = ChatPromptTemplate.from_template('{text}를 한 줄로 요약해주세요'),
    keywords = ChatPromptTemplate.from_template('{text}에서 키워드를 3개만 뽑아주세요')
)

In [3]:
result = parallel_chain.invoke({"text" : "Langchain은 LLM 기반 application 개발 프레임워크입니다."})

In [4]:
result

{'summary': ChatPromptValue(messages=[HumanMessage(content='Langchain은 LLM 기반 application 개발 프레임워크입니다.를 한 줄로 요약해주세요', additional_kwargs={}, response_metadata={})]),
 'keywords': ChatPromptValue(messages=[HumanMessage(content='Langchain은 LLM 기반 application 개발 프레임워크입니다.에서 키워드를 3개만 뽑아주세요', additional_kwargs={}, response_metadata={})])}

In [7]:
llm = ChatOpenAI(model = "gpt-4o-mini")

In [32]:
# RunnableBranch
from langchain_core.runnables import RunnableBranch


parser = StrOutputParser()
tech_chain =ChatPromptTemplate.from_template('기술지원팀입니다 : {question}') | llm | parser
billing_chain = ChatPromptTemplate.from_template('요금 관리 팀입니다 : {question}') | llm | parser
general_chain = ChatPromptTemplate.from_template('일반 상담 팀입니다 : {question}') | llm | parser

def route_logic(x):
    text = x['question']   # ~~~쓰다가 에러가 났어요
    if( '오류' in text) or ('에러' in text) : 
        return 'technical'
    elif '가격' in text or '요금' in text :
        return 'billing'
    return 'general'

branch = RunnableBranch(
    (lambda x : route_logic(x) == "technical", tech_chain),
    (lambda x : route_logic(x) == "billing", billing_chain),
    general_chain
)

questions = ["프린터 오류가 났습니다", "월 요금이 얼마인가요?", "영업시간 알려주세요"]
for q in questions:
    print(f"Q: {q}\nA: {branch.invoke({'question':q})}\n")

Q: 프린터 오류가 났습니다
A: 안녕하세요! 프린터 오류에 대해 도와드리겠습니다. 어떤 오류 메시지가 나타나는지, 또는 어떤 증상이 있는지 자세히 말씀해 주시면 더욱 정확한 지원을 해드릴 수 있습니다. 프린터의 모델명과 사용 중인 컴퓨터 운영 체제도 알려주시면 좋습니다.

Q: 월 요금이 얼마인가요?
A: 안녕하세요! 요금 관리 팀입니다. 월 요금은 서비스 종류와 사용량에 따라 다르게 적용됩니다. 정확한 요금을 알고 싶으시다면 사용하고 계시는 서비스의 종류와 세부 정보를 말씀해 주시면 더 자세히 안내해 드리겠습니다.

Q: 영업시간 알려주세요
A: 안녕하세요! 일반 상담 팀입니다. 영업시간은 보통 월요일부터 금요일까지 오전 9시부터 오후 6시까지입니다. 특정한 질문이나 추가적인 정보가 필요하시면 언제든지 말씀해 주세요!



In [10]:
llm.invoke("안녕하세요")

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 9, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1681c17ec', 'id': 'chatcmpl-DJ7vZdzyuJc1g29rx2eNlk6frbWUy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce9eb-e30b-75a0-a86f-e8389d848388-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 10, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
!pip install gradio

In [11]:
import gradio as gr

In [16]:
demo1.close()

Closing server running on port: 7860


In [15]:
def greet(name):
    return f"안녕하세요, {name}님"

demo1 = gr.Interface(
    fn = greet,
    inputs = gr.Textbox(label='이름 입력'),
    outputs = gr.Textbox(label = '인사말'),
    title = '인사봇'
)

demo1.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://951566b81a62a54691.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
def echo_bot(message, history):
    return f"Echo : {message}"

demo2 = gr.ChatInterface(
    fn = echo_bot,
    title = '에코챗봇',
    examples = ["안녕하세요", "오늘 날씨 어때요", "FAQ 챗봇 테스트"]
)

demo2.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ccfce9ef6e21b6d50e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:
with gr.Blocks(title = 'custom layout') as demo3:
    gr.Markdown('custom layout demo')
    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(label='질문')
            submit_btn = gr.Button("전송")
        with gr.Column(scale=1):
            category_output = gr.Textbox(label='카테고리')
            
    output_text = gr.Textbox(label='답변')
    
    def process(text):
        cat = "기술" if any(kw in text for kw in ["오류", "설치", "연결"]) else "일반"

        return cat, f"[{cat}] {text}"
    
    submit_btn.click(fn=process, inputs=input_text, outputs=[category_output, output_text])

demo3.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://c22834912fc51cff0c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
demo3.close()

Closing server running on port: 7862


In [28]:
history = [('a',1), ('b',2), ('c',3)]
for human, ai in history:
    print(human, ai)

a 1
b 2
c 3


In [29]:
history = [{'role': 'user', 'content':'abcde'}, {'role': 'user1', 'content':'abcdef'}, {'role': 'user2', 'content':'abcdefg'}]
for msg in history:
    print(msg['role'], msg['content'])

user abcde
user1 abcdef
user2 abcdefg


In [30]:
llm = ChatOpenAI(model = "gpt-4o-mini")
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

def echo_bot(message, history):
    return f"Echo : {message}"

def chat_with_llm(message, history):
    messages = [SystemMessage(content ="당신은 친절한 한국어 어시스턴트입니다.")]
#     for human, ai in history:
#         messages.append(HumanMessage(content=human))
#         messages.append(AIMessage(content=ai))
    for mmm in history:
        if mmm['role'] == 'user':
            messages.append(HumanMessage(content=mmm['content']))
        else:
            messages.append(AIMessage(content=mmm['content']))
    
    messages.append(HumanMessage(content=message))
    return llm.invoke(messages).content
    

demo = gr.ChatInterface(
    fn = chat_with_llm,
    title = 'Chatbot',
    examples = ["안녕하세요", "Python에 대해 알려주세요", "오늘 기분이 좋아요"]
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://a2f5ca5360fa18ce8b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [35]:
faq_data = [
    {"category": "계정", "question": "비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?",
     "answer": "IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다."},
    {"category": "계정", "question": "계정이 잠겼습니다. 어떻게 해제하나요?",
     "answer": "5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요."},
    {"category": "계정", "question": "신규 계정은 어떻게 만드나요?",
     "answer": "신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다."},
    {"category": "계정", "question": "2단계 인증(MFA)을 설정하려면?",
     "answer": "IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요."},
]

llm = ChatOpenAI(model = "gpt-4o-mini")

faq_context = '\n'.join([f"Q : {item['question']}\nA: {item['answer']}" for item in faq_data])

In [36]:
faq_context

"Q : 비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?\nA: IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다.\nQ : 계정이 잠겼습니다. 어떻게 해제하나요?\nA: 5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요.\nQ : 신규 계정은 어떻게 만드나요?\nA: 신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다.\nQ : 2단계 인증(MFA)을 설정하려면?\nA: IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요."

In [38]:
prompt = ChatPromptTemplate.from_messages([
    ("system", f"너는 사내 IT 지원팀 챗봇이야. 아래 제공된 FAQ 데이터를 바탕으로 사용자 질문에 친절하게 답해줘. "
                f"데이터에 없는 내용은 'IT 헬프데스크(1234번)으로 문의해주세요' 라고 안내해줘\n\n[FAQ데이터]\n{faq_context}"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

def chat_response(message, history):
    response = chain.invoke({"question" : message})
    return response

demo = gr.ChatInterface(
    fn = chat_response,
    title = '사내지원챗봇',
    examples = ["비밀번호 어떻게 초기화하나요", "Wi-fi가 너무 느려요", "2단계 인증 어떻게 설정하나요"],
    description = "무엇을 도와드릴까요?"
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://8049a5722e17c164e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
